# 20. RAW・Platt・Isotonicの確率校正
出典: FX (2).ipynb、セルindex [42, 43]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 42


In [ ]:
# ============================================================
# USD/JPY
# NESTED PROBABILITY CALIBRATION TEST
#
# Random Forest
# + Probability Calibration
# + Confidence Threshold
# + Session Filter
# + FIXED 30 MIN EXIT
#
# ============================================================
#
# 検証対象
#
# 1. RAW probability
# 2. Platt Scaling
# 3. Isotonic Regression
#
#
# 目的
#
# RandomForestが出す
#
#   p_up = 0.60
#
# が、本当に約60%の上昇確率を意味するのか確認する。
#
#
# 重要:
#
# Calibration methodはTest年を見ずに
# Validationだけで決定する。
#
# Exitは30分固定。
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)


# ============================================================
# 0. 必要な前コード確認
# ============================================================

REQUIRED_OBJECTS = [

    "data",
    "FEATURES",

    "build_model",
    "predict_frame",

    "choose_threshold",
    "choose_session",

    "select_trades",
    "strategy_stats",

    "BASE_COST",

    "MIN_TRAIN_YEARS",
    "MIN_TRAIN_ROWS",
    "MIN_EVAL_ROWS",
]


missing = [

    name

    for name in REQUIRED_OBJECTS

    if name not in globals()
]


if missing:

    print(
        "前のSession / 30分固定検証コードを先に実行してください。"
    )

    print(
        "不足:",
        missing
    )

    raise RuntimeError(
        "Required previous objects are missing."
    )


# ============================================================
# 1. CONFIG
# ============================================================

EPS = 1e-6


# ------------------------------------------------------------
# Calibrationに最低限必要なOOFサンプル数
# ------------------------------------------------------------

MIN_CALIBRATION_ROWS = 5000


# ------------------------------------------------------------
# Isotonicはデータが少ないと過学習しやすいので、
# こちらはさらに厳しくする
# ------------------------------------------------------------

MIN_ISOTONIC_ROWS = 10000


# ------------------------------------------------------------
# Calibration band
# ------------------------------------------------------------

CALIBRATION_BINS = [

    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    0.80,
    0.90,
    1.000001,
]


CALIBRATION_LABELS = [

    "50-52",
    "52-54",
    "54-56",
    "56-58",
    "58-60",
    "60-62",
    "62-65",
    "65-70",
    "70-80",
    "80-90",
    "90-100",
]


OUTPUT_DIR = (

    Path.cwd()

    /

    (
        "nested_probability_calibration_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )
)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. Probability utility
# ============================================================

def clip_probability(p):

    return np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        EPS,
        1 - EPS,
    )


def probability_logit(p):

    p = clip_probability(
        p
    )

    return np.log(
        p
        /
        (
            1 - p
        )
    )


# ============================================================
# 3. Platt Scaling
# ============================================================

class PlattCalibrator:

    def __init__(self):

        self.model = LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
        )


    def fit(
        self,
        probability,
        target,
    ):

        x = probability_logit(
            probability
        ).reshape(
            -1,
            1
        )

        y = np.asarray(
            target,
            dtype=int
        )


        self.model.fit(
            x,
            y
        )


        return self


    def predict(
        self,
        probability,
    ):

        x = probability_logit(
            probability
        ).reshape(
            -1,
            1
        )


        return (
            self.model
            .predict_proba(
                x
            )[:, 1]
        )


# ============================================================
# 4. Isotonic Calibration
# ============================================================

class IsotonicCalibrator:

    def __init__(self):

        self.model = IsotonicRegression(
            out_of_bounds="clip"
        )


    def fit(
        self,
        probability,
        target,
    ):

        self.model.fit(
            np.asarray(
                probability,
                dtype=float
            ),

            np.asarray(
                target,
                dtype=int
            ),
        )


        return self


    def predict(
        self,
        probability,
    ):

        result = self.model.predict(
            np.asarray(
                probability,
                dtype=float
            )
        )


        return clip_probability(
            result
        )


# ============================================================
# 5. RAW
# ============================================================

class RawCalibrator:

    def fit(
        self,
        probability,
        target,
    ):

        return self


    def predict(
        self,
        probability,
    ):

        return clip_probability(
            probability
        )


# ============================================================
# 6. Calibration method factory
# ============================================================

def make_calibrator(
    name
):

    if name == "RAW":

        return RawCalibrator()


    if name == "PLATT":

        return PlattCalibrator()


    if name == "ISOTONIC":

        return IsotonicCalibrator()


    raise ValueError(
        name
    )


# ============================================================
# 7. predict_frameのp_upをCalibration後に置き換える
# ============================================================

def apply_calibrated_probability(
    prediction_frame,
    calibrated_p_up,
):

    result = prediction_frame.copy()


    p_up = clip_probability(
        calibrated_p_up
    )


    result[
        "p_up"
    ] = p_up


    result[
        "p_down"
    ] = (
        1
        -
        p_up
    )


    # --------------------------------------------------------
    # Direction
    # --------------------------------------------------------

    result[
        "direction"
    ] = np.where(

        result[
            "p_up"
        ]

        >=

        result[
            "p_down"
        ],

        "BUY",

        "SELL",
    )


    # --------------------------------------------------------
    # Confidence
    #
    # BUYならp_up
    # SELLならp_down
    # --------------------------------------------------------

    result[
        "confidence"
    ] = np.maximum(

        result[
            "p_up"
        ],

        result[
            "p_down"
        ],
    )


    return result


# ============================================================
# 8. Calibration error
#
# Expected Calibration Error
# ============================================================

def expected_calibration_error(
    probability,
    target,
    bins=10,
):

    probability = np.asarray(
        probability,
        dtype=float
    )

    target = np.asarray(
        target,
        dtype=int
    )


    edges = np.linspace(
        0,
        1,
        bins + 1,
    )


    total = len(
        probability
    )


    error = 0.0


    for i in range(
        bins
    ):

        left = edges[i]
        right = edges[i + 1]


        if i == bins - 1:

            mask = (
                (probability >= left)
                &
                (probability <= right)
            )

        else:

            mask = (
                (probability >= left)
                &
                (probability < right)
            )


        n = mask.sum()


        if n == 0:

            continue


        mean_probability = (
            probability[
                mask
            ].mean()
        )


        actual_rate = (
            target[
                mask
            ].mean()
        )


        error += (

            n
            /
            total

            *

            abs(
                mean_probability
                -
                actual_rate
            )
        )


    return error


# ============================================================
# 9. Calibration score
# ============================================================

def calibration_metrics(
    probability,
    target,
):

    probability = clip_probability(
        probability
    )


    target = np.asarray(
        target,
        dtype=int
    )


    return {

        "brier_score":
            brier_score_loss(
                target,
                probability
            ),

        "ece":
            expected_calibration_error(
                probability,
                target
            ),

        "auc":
            roc_auc_score(
                target,
                probability
            ),
    }


# ============================================================
# 10. Calibration Band
#
# Direction confidenceで評価
# ============================================================

def calibration_band_table(
    probability,
    target,
):

    probability = clip_probability(
        probability
    )


    target = np.asarray(
        target,
        dtype=int
    )


    direction = np.where(
        probability >= 0.5,
        1,
        0,
    )


    confidence = np.maximum(
        probability,
        1 - probability,
    )


    correct = (
        direction
        ==
        target
    )


    temp = pd.DataFrame(

        {

            "confidence":
                confidence,

            "correct":
                correct.astype(
                    float
                ),
        }

    )


    temp[
        "confidence_band"
    ] = pd.cut(

        temp[
            "confidence"
        ],

        bins=
            CALIBRATION_BINS,

        labels=
            CALIBRATION_LABELS,

        right=False,
    )


    rows = []


    for band, group in temp.groupby(
        "confidence_band",
        observed=True
    ):

        if len(group) == 0:

            continue


        rows.append(

            {

                "confidence_band":
                    str(
                        band
                    ),

                "samples":
                    len(
                        group
                    ),

                "mean_confidence":
                    group[
                        "confidence"
                    ].mean(),

                "actual_accuracy":
                    group[
                        "correct"
                    ].mean(),

                "calibration_gap":
                    (
                        group[
                            "actual_accuracy"
                        ].mean()

                        -

                        group[
                            "confidence"
                        ].mean()
                    ),
            }

        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 11. 年単位Walk-Forward OOF予測を生成
#
# Calibration用データを作る。
#
# 未来情報を使わない。
# ============================================================

def build_oof_predictions(
    source_data,
):

    source_years = sorted(
        source_data.index.year.unique()
    )


    frames = []


    for prediction_year in source_years:


        year_start = pd.Timestamp(
            year=prediction_year,
            month=1,
            day=1,
            tz="UTC",
        )


        year_end = pd.Timestamp(
            year=prediction_year + 1,
            month=1,
            day=1,
            tz="UTC",
        )


        train_part = source_data.loc[

            (
                source_data.index
                <
                year_start
            )

            &

            (
                source_data[
                    "label_end"
                ]
                <=
                year_start
            )

        ].copy()


        eval_part = source_data.loc[

            (
                source_data.index
                >=
                year_start
            )

            &

            (
                source_data.index
                <
                year_end
            )

            &

            (
                source_data[
                    "label_end"
                ]
                <=
                year_end
            )

        ].copy()


        # ----------------------------------------------------
        # 最低Train量
        # ----------------------------------------------------

        if (
            len(train_part)
            <
            MIN_TRAIN_ROWS
        ):

            continue


        if (
            len(eval_part)
            <
            MIN_EVAL_ROWS
        ):

            continue


        model = build_model()


        model.fit(

            train_part[
                FEATURES
            ],

            train_part[
                "target"
            ],
        )


        prediction = predict_frame(

            model,

            eval_part,
        )


        frame = pd.DataFrame(

            {

                "target":
                    eval_part[
                        "target"
                    ],

                "raw_p_up":
                    prediction[
                        "p_up"
                    ],
            },

            index=
                eval_part.index,

        )


        frame[
            "prediction_year"
        ] = (
            prediction_year
        )


        frames.append(
            frame
        )


    if not frames:

        return pd.DataFrame()


    result = (

        pd.concat(
            frames
        )

        .sort_index()
    )


    return result


# ============================================================
# 12. Calibration modelsをOOFで作る
# ============================================================

def fit_calibrators_from_oof(
    train_data
):

    oof = build_oof_predictions(
        train_data
    )


    if (
        len(oof)
        <
        MIN_CALIBRATION_ROWS
    ):

        print(
            "Calibration OOFが不足:",
            len(oof)
        )


        return (
            {
                "RAW":
                    RawCalibrator()
            },

            oof,
        )


    calibrators = {}


    # --------------------------------------------------------
    # RAW
    # --------------------------------------------------------

    raw = RawCalibrator()

    raw.fit(
        oof[
            "raw_p_up"
        ],

        oof[
            "target"
        ],
    )


    calibrators[
        "RAW"
    ] = raw


    # --------------------------------------------------------
    # PLATT
    # --------------------------------------------------------

    platt = PlattCalibrator()

    platt.fit(

        oof[
            "raw_p_up"
        ],

        oof[
            "target"
        ],
    )


    calibrators[
        "PLATT"
    ] = platt


    # --------------------------------------------------------
    # ISOTONIC
    # --------------------------------------------------------

    if (
        len(oof)
        >=
        MIN_ISOTONIC_ROWS
    ):

        isotonic = IsotonicCalibrator()

        isotonic.fit(

            oof[
                "raw_p_up"
            ],

            oof[
                "target"
            ],
        )


        calibrators[
            "ISOTONIC"
        ] = isotonic


    return (
        calibrators,
        oof,
    )


# ============================================================
# 13. ValidationでCalibration方式を選択
#
# 第一評価 = Brier
# 第二評価 = ECE
# ------------------------------------------------------------
#
# AUCはCalibrationでは基本的に大きく変わらない。
# Calibrationの目的は順位ではなく
# 「確率の正しさ」。
# ============================================================

def choose_calibration_method(
    calibrators,
    validation_predictions,
    validation_target,
):

    rows = []


    raw_probability = (
        validation_predictions[
            "p_up"
        ].to_numpy()
    )


    for name, calibrator in calibrators.items():

        probability = calibrator.predict(
            raw_probability
        )


        metrics = calibration_metrics(
            probability,
            validation_target,
        )


        rows.append(

            {

                "method":
                    name,

                **metrics,
            }

        )


    table = pd.DataFrame(
        rows
    )


    if table.empty:

        return (
            "RAW",
            table,
        )


    # --------------------------------------------------------
    # Brier最小
    # ↓
    # ECE最小
    # --------------------------------------------------------

    selected = (

        table

        .sort_values(

            [
                "brier_score",
                "ece",
            ],

            ascending=[
                True,
                True,
            ],

        )

        .iloc[0]

    )


    return (
        selected[
            "method"
        ],

        table,
    )


# ============================================================
# 14. Nested Walk Forward
# ============================================================

years = sorted(
    data.index.year.unique()
)


annual_rows = []

calibration_score_rows = []

calibration_band_frames = []

raw_oos_frames = []

calibrated_oos_frames = []

selected_trade_frames = []


for test_year in years:


    validation_year = (
        test_year
        -
        1
    )


    previous_years = [

        y

        for y in years

        if y
        <
        validation_year

    ]


    if (
        len(previous_years)
        <
        MIN_TRAIN_YEARS
    ):

        continue


    if validation_year not in years:

        continue


    validation_start = pd.Timestamp(

        year=
            validation_year,

        month=1,

        day=1,

        tz="UTC",
    )


    test_start = pd.Timestamp(

        year=
            test_year,

        month=1,

        day=1,

        tz="UTC",
    )


    test_end = pd.Timestamp(

        year=
            test_year + 1,

        month=1,

        day=1,

        tz="UTC",
    )


    # ========================================================
    # TRAIN
    # ========================================================

    train = data.loc[

        (
            data.index
            <
            validation_start
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            validation_start
        )

    ].copy()


    # ========================================================
    # VALIDATION
    # ========================================================

    validation = data.loc[

        (
            data.index
            >=
            validation_start
        )

        &

        (
            data.index
            <
            test_start
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            test_start
        )

    ].copy()


    # ========================================================
    # FINAL TRAIN
    # ========================================================

    final_train = data.loc[

        (
            data.index
            <
            test_start
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            test_start
        )

    ].copy()


    # ========================================================
    # TEST
    # ========================================================

    test = data.loc[

        (
            data.index
            >=
            test_start
        )

        &

        (
            data.index
            <
            test_end
        )

        &

        (
            data[
                "label_end"
            ]
            <=
            test_end
        )

    ].copy()


    if (
        len(train)
        <
        MIN_TRAIN_ROWS

        or

        len(validation)
        <
        MIN_EVAL_ROWS

        or

        len(final_train)
        <
        MIN_TRAIN_ROWS

        or

        len(test)
        <
        MIN_EVAL_ROWS
    ):

        continue


    print()
    print("=" * 60)
    print(
        "CALIBRATION TEST YEAR",
        test_year
    )
    print("=" * 60)


    # ========================================================
    # 15. Trainモデル
    # ========================================================

    base_model = build_model()


    base_model.fit(

        train[
            FEATURES
        ],

        train[
            "target"
        ],
    )


    validation_predictions_raw = predict_frame(

        base_model,

        validation,
    )


    # ========================================================
    # 16. Train OOFだけでCalibratorを作成
    # ========================================================

    calibrators_validation, oof_train = (

        fit_calibrators_from_oof(
            train
        )

    )


    print(
        "Train OOF rows:",
        len(
            oof_train
        )
    )


    # ========================================================
    # 17. Calibration MethodをValidationで選択
    # ========================================================

    selected_method, calibration_table = (

        choose_calibration_method(

            calibrators_validation,

            validation_predictions_raw,

            validation[
                "target"
            ],
        )

    )


    calibration_table[
        "test_year"
    ] = test_year


    calibration_score_rows.append(
        calibration_table
    )


    print()
    print(
        "Calibration comparison:"
    )

    print(
        calibration_table.to_string(
            index=False
        )
    )


    print()
    print(
        "Selected Calibration:",
        selected_method
    )


    # ========================================================
    # 18. Validation calibrated probability
    # ========================================================

    chosen_validation_calibrator = (

        calibrators_validation[
            selected_method
        ]

    )


    validation_calibrated_probability = (

        chosen_validation_calibrator.predict(

            validation_predictions_raw[
                "p_up"
            ]
        )

    )


    validation_predictions = (

        apply_calibrated_probability(

            validation_predictions_raw,

            validation_calibrated_probability,
        )

    )


    # ========================================================
    # 19. ThresholdをCalibration後確率で選択
    # ========================================================

    threshold_choice, _ = (

        choose_threshold(

            validation_predictions

        )

    )


    if threshold_choice is None:

        continue


    threshold = (

        threshold_choice[
            "threshold"
        ]

    )


    # ========================================================
    # 20. Session
    # ========================================================

    session_choice, _ = (

        choose_session(

            validation_predictions,

            threshold,
        )

    )


    if session_choice is None:

        continue


    session = (

        session_choice[
            "session_policy"
        ]

    )


    print(
        "Threshold:",
        threshold
    )

    print(
        "Session:",
        session
    )


    # ========================================================
    # 21. Test用Calibrator
    #
    # final_trainまでのOOFだけを使用。
    #
    # Test年は一切使用しない。
    # ========================================================

    calibrators_test, oof_final = (

        fit_calibrators_from_oof(
            final_train
        )

    )


    # --------------------------------------------------------
    # Validationで選ばれた方法が、
    # OOF不足などで存在しない場合RAWへ戻す
    # --------------------------------------------------------

    if (
        selected_method
        not in calibrators_test
    ):

        actual_test_method = (
            "RAW"
        )

    else:

        actual_test_method = (
            selected_method
        )


    # ========================================================
    # 22. Final model
    # ========================================================

    final_model = build_model()


    final_model.fit(

        final_train[
            FEATURES
        ],

        final_train[
            "target"
        ],
    )


    test_predictions_raw = predict_frame(

        final_model,

        test,
    )


    # ========================================================
    # 23. RAW OOS probability
    # ========================================================

    raw_probability = (

        test_predictions_raw[
            "p_up"
        ].to_numpy()

    )


    raw_metrics = calibration_metrics(

        raw_probability,

        test[
            "target"
        ],
    )


    # ========================================================
    # 24. Calibrated OOS probability
    # ========================================================

    selected_calibrator = (

        calibrators_test[
            actual_test_method
        ]

    )


    calibrated_probability = (

        selected_calibrator.predict(

            raw_probability

        )

    )


    calibrated_metrics = calibration_metrics(

        calibrated_probability,

        test[
            "target"
        ],
    )


    # ========================================================
    # 25. Calibration Band
    # ========================================================

    raw_band = calibration_band_table(

        raw_probability,

        test[
            "target"
        ],
    )


    raw_band[
        "method"
    ] = "RAW"


    raw_band[
        "test_year"
    ] = test_year


    calibrated_band = calibration_band_table(

        calibrated_probability,

        test[
            "target"
        ],
    )


    calibrated_band[
        "method"
    ] = actual_test_method


    calibrated_band[
        "test_year"
    ] = test_year


    calibration_band_frames.extend(

        [
            raw_band,
            calibrated_band,
        ]

    )


    # ========================================================
    # 26. StrategyにCalibrationを適用
    # ========================================================

    test_predictions_calibrated = (

        apply_calibrated_probability(

            test_predictions_raw,

            calibrated_probability,
        )

    )


    calibrated_trades = (

        select_trades(

            test_predictions_calibrated,

            threshold=
                threshold,

            session_policy=
                session,

            cost=
                BASE_COST,
        )

    )


    # ========================================================
    # RAW Strategy比較
    #
    # 同じThreshold / SessionをRAWにも適用し、
    # Calibrationだけの影響を見る。
    # ========================================================

    raw_trades = (

        select_trades(

            test_predictions_raw,

            threshold=
                threshold,

            session_policy=
                session,

            cost=
                BASE_COST,
        )

    )


    # ========================================================
    # 27. Strategy Stats
    # ========================================================

    if len(
        calibrated_trades
    ):

        calibrated_stats = strategy_stats(

            calibrated_trades[
                "net_return"
            ]

        )

    else:

        calibrated_stats = {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "median_return": np.nan,
            "profit_factor": np.nan,
            "max_dd": np.nan,
            "growth": np.nan,
        }


    if len(
        raw_trades
    ):

        raw_stats = strategy_stats(

            raw_trades[
                "net_return"
            ]

        )

    else:

        raw_stats = {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "median_return": np.nan,
            "profit_factor": np.nan,
            "max_dd": np.nan,
            "growth": np.nan,
        }


    # ========================================================
    # 28. 保存用OOS
    # ========================================================

    raw_oos = pd.DataFrame(

        {

            "target":
                test[
                    "target"
                ],

            "p_up":
                raw_probability,

            "test_year":
                test_year,
        },

        index=
            test.index,

    )


    calibrated_oos = pd.DataFrame(

        {

            "target":
                test[
                    "target"
                ],

            "p_up":
                calibrated_probability,

            "method":
                actual_test_method,

            "test_year":
                test_year,
        },

        index=
            test.index,

    )


    raw_oos_frames.append(
        raw_oos
    )


    calibrated_oos_frames.append(
        calibrated_oos
    )


    if len(
        calibrated_trades
    ):

        temp_trades = calibrated_trades.copy()

        temp_trades[
            "test_year"
        ] = test_year

        temp_trades[
            "calibration_method"
        ] = actual_test_method

        selected_trade_frames.append(
            temp_trades
        )


    # ========================================================
    # 29. Annual row
    # ========================================================

    annual_rows.append(

        {

            "test_year":
                test_year,

            "calibration_method":
                actual_test_method,

            "threshold":
                threshold,

            "session_policy":
                session,

            "raw_brier":
                raw_metrics[
                    "brier_score"
                ],

            "cal_brier":
                calibrated_metrics[
                    "brier_score"
                ],

            "raw_ece":
                raw_metrics[
                    "ece"
                ],

            "cal_ece":
                calibrated_metrics[
                    "ece"
                ],

            "raw_auc":
                raw_metrics[
                    "auc"
                ],

            "cal_auc":
                calibrated_metrics[
                    "auc"
                ],

            "raw_trades":
                len(
                    raw_trades
                ),

            "cal_trades":
                len(
                    calibrated_trades
                ),

            "raw_pf":
                raw_stats[
                    "profit_factor"
                ],

            "cal_pf":
                calibrated_stats[
                    "profit_factor"
                ],

            "raw_avg":
                raw_stats[
                    "avg_return"
                ],

            "cal_avg":
                calibrated_stats[
                    "avg_return"
                ],

            "raw_dd":
                raw_stats[
                    "max_dd"
                ],

            "cal_dd":
                calibrated_stats[
                    "max_dd"
                ],
        }

    )


    print()
    print(
        "Test RAW Brier:",
        round(
            raw_metrics[
                "brier_score"
            ],
            6
        )
    )


    print(
        "Test CAL Brier:",
        round(
            calibrated_metrics[
                "brier_score"
            ],
            6
        )
    )


    print(
        "RAW ECE:",
        round(
            raw_metrics[
                "ece"
            ],
            6
        )
    )


    print(
        "CAL ECE:",
        round(
            calibrated_metrics[
                "ece"
            ],
            6
        )
    )


    print(
        "RAW PF:",
        round(
            raw_stats[
                "profit_factor"
            ],
            3
        )
    )


    print(
        "CAL PF:",
        round(
            calibrated_stats[
                "profit_factor"
            ],
            3
        )
    )


# ============================================================
# 30. Annual Results
# ============================================================

annual_results = pd.DataFrame(
    annual_rows
)


print()
print("=" * 70)
print("ANNUAL CALIBRATION RESULTS")
print("=" * 70)


annual_show = annual_results.copy()


for col in [

    "threshold",

    "raw_avg",
    "cal_avg",

    "raw_dd",
    "cal_dd",

]:

    if col in annual_show.columns:

        annual_show[
            col
        ] *= 100


print(

    annual_show.to_string(
        index=False
    )

)


# ============================================================
# 31. Method Selection Frequency
# ============================================================

print()
print("=" * 70)
print("CALIBRATION METHOD FREQUENCY")
print("=" * 70)


print(

    annual_results[
        "calibration_method"
    ]

    .value_counts()

)


# ============================================================
# 32. Probability quality improvement
# ============================================================

annual_results[
    "brier_improved"
] = (

    annual_results[
        "cal_brier"
    ]

    <

    annual_results[
        "raw_brier"
    ]

)


annual_results[
    "ece_improved"
] = (

    annual_results[
        "cal_ece"
    ]

    <

    annual_results[
        "raw_ece"
    ]

)


print()
print("=" * 70)
print("CALIBRATION VALUE")
print("=" * 70)


print(
    "Brier improved:",
    annual_results[
        "brier_improved"
    ].sum(),
    "/",
    len(
        annual_results
    )
)


print(
    "ECE improved:",
    annual_results[
        "ece_improved"
    ].sum(),
    "/",
    len(
        annual_results
    )
)


print()


print(
    "Mean RAW Brier:",
    annual_results[
        "raw_brier"
    ].mean()
)


print(
    "Mean CAL Brier:",
    annual_results[
        "cal_brier"
    ].mean()
)


print()


print(
    "Mean RAW ECE:",
    annual_results[
        "raw_ece"
    ].mean()
)


print(
    "Mean CAL ECE:",
    annual_results[
        "cal_ece"
    ].mean()
)


# ============================================================
# 33. Overall OOS Probability
# ============================================================

all_raw_oos = (

    pd.concat(
        raw_oos_frames
    )

    .sort_index()

)


all_calibrated_oos = (

    pd.concat(
        calibrated_oos_frames
    )

    .sort_index()

)


overall_raw_metrics = calibration_metrics(

    all_raw_oos[
        "p_up"
    ],

    all_raw_oos[
        "target"
    ],
)


overall_cal_metrics = calibration_metrics(

    all_calibrated_oos[
        "p_up"
    ],

    all_calibrated_oos[
        "target"
    ],
)


print()
print("=" * 70)
print("OVERALL OOS PROBABILITY QUALITY")
print("=" * 70)


print(
    "RAW"
)

print(
    overall_raw_metrics
)


print()

print(
    "CALIBRATED"
)

print(
    overall_cal_metrics
)


# ============================================================
# 34. Overall Calibration bands
# ============================================================

overall_raw_band = calibration_band_table(

    all_raw_oos[
        "p_up"
    ],

    all_raw_oos[
        "target"
    ],
)


overall_cal_band = calibration_band_table(

    all_calibrated_oos[
        "p_up"
    ],

    all_calibrated_oos[
        "target"
    ],
)


print()
print("=" * 70)
print("RAW CONFIDENCE CALIBRATION")
print("=" * 70)


raw_band_show = overall_raw_band.copy()


for col in [

    "mean_confidence",
    "actual_accuracy",
    "calibration_gap",

]:

    raw_band_show[
        col
    ] *= 100


print(

    raw_band_show.to_string(
        index=False
    )

)


print()
print("=" * 70)
print("CALIBRATED CONFIDENCE")
print("=" * 70)


cal_band_show = overall_cal_band.copy()


for col in [

    "mean_confidence",
    "actual_accuracy",
    "calibration_gap",

]:

    cal_band_show[
        col
    ] *= 100


print(

    cal_band_show.to_string(
        index=False
    )

)


# ============================================================
# 35. Confidence帯別Strategy
#
# Calibration後Probabilityのみ。
#
# Position sizingへ進むための最重要表。
# ============================================================

if selected_trade_frames:

    all_selected_trades = (

        pd.concat(
            selected_trade_frames
        )

        .sort_index()

    )


    all_selected_trades[
        "confidence_band"
    ] = pd.cut(

        all_selected_trades[
            "confidence"
        ],

        bins=
            CALIBRATION_BINS,

        labels=
            CALIBRATION_LABELS,

        right=False,
    )


    confidence_strategy_rows = []


    for band, group in (

        all_selected_trades.groupby(
            "confidence_band",
            observed=True
        )

    ):


        if len(
            group
        ) == 0:

            continue


        stats = strategy_stats(

            group[
                "net_return"
            ]

        )


        confidence_strategy_rows.append(

            {

                "confidence_band":
                    str(
                        band
                    ),

                "trades":
                    len(
                        group
                    ),

                "mean_confidence":
                    group[
                        "confidence"
                    ].mean(),

                **stats,
            }

        )


    confidence_strategy = pd.DataFrame(

        confidence_strategy_rows

    )


    print()
    print("=" * 70)
    print("CALIBRATED CONFIDENCE x PROFIT")
    print("=" * 70)


    strategy_show = confidence_strategy.copy()


    for col in [

        "mean_confidence",
        "win_rate",
        "avg_return",
        "median_return",
        "max_dd",
        "growth",

    ]:

        if col in strategy_show.columns:

            strategy_show[
                col
            ] *= 100


    print(

        strategy_show.to_string(
            index=False
        )

    )


else:

    all_selected_trades = pd.DataFrame()

    confidence_strategy = pd.DataFrame()


# ============================================================
# 36. Strategy comparison
# ============================================================

annual_results[
    "pf_improved"
] = (

    annual_results[
        "cal_pf"
    ]

    >

    annual_results[
        "raw_pf"
    ]

)


annual_results[
    "avg_improved"
] = (

    annual_results[
        "cal_avg"
    ]

    >

    annual_results[
        "raw_avg"
    ]

)


print()
print("=" * 70)
print("CALIBRATION STRATEGY EFFECT")
print("=" * 70)


print(
    "PF improved:",
    annual_results[
        "pf_improved"
    ].sum(),
    "/",
    len(
        annual_results
    )
)


print(
    "Avg Return improved:",
    annual_results[
        "avg_improved"
    ].sum(),
    "/",
    len(
        annual_results
    )
)


# ============================================================
# 37. Automatic Decision
# ============================================================

print()
print("=" * 70)
print("AUTOMATIC DECISION")
print("=" * 70)


brier_better = (

    annual_results[
        "brier_improved"
    ].sum()

)


ece_better = (

    annual_results[
        "ece_improved"
    ].sum()

)


n_years = len(
    annual_results
)


print(
    "Years:",
    n_years
)


print(
    "Brier better:",
    brier_better,
    "/",
    n_years
)


print(
    "ECE better:",
    ece_better,
    "/",
    n_years
)


print()


print(
    "Overall RAW Brier:",
    overall_raw_metrics[
        "brier_score"
    ]
)


print(
    "Overall CAL Brier:",
    overall_cal_metrics[
        "brier_score"
    ]
)


print()


print(
    "Overall RAW ECE:",
    overall_raw_metrics[
        "ece"
    ]
)


print(
    "Overall CAL ECE:",
    overall_cal_metrics[
        "ece"
    ]
)


print()


if (

    overall_cal_metrics[
        "brier_score"
    ]

    <

    overall_raw_metrics[
        "brier_score"
    ]

    and

    overall_cal_metrics[
        "ece"
    ]

    <

    overall_raw_metrics[
        "ece"
    ]

    and

    brier_better
    >=
    4

):

    print(
        "判定: PROBABILITY CALIBRATION HAS OOS VALUE"
    )

    print()

    print(
        "Calibration後Confidenceを"
    )

    print(
        "Position Sizingに使用する価値があります。"
    )


else:

    print(
        "判定: RAW PROBABILITY REMAINS PREFERABLE"
    )

    print()

    print(
        "無理にCalibrationせず、RAW Confidenceを維持します。"
    )


# ============================================================
# 38. Calibration Plot
# ============================================================

plt.figure(
    figsize=(
        7,
        6
    )
)


plt.plot(
    [
        0.5,
        1.0
    ],
    [
        0.5,
        1.0
    ],
    linestyle="--",
)


if not overall_raw_band.empty:

    plt.plot(

        overall_raw_band[
            "mean_confidence"
        ],

        overall_raw_band[
            "actual_accuracy"
        ],

        marker="o",

        label=
            "RAW",
    )


if not overall_cal_band.empty:

    plt.plot(

        overall_cal_band[
            "mean_confidence"
        ],

        overall_cal_band[
            "actual_accuracy"
        ],

        marker="o",

        label=
            "Calibrated",
    )


plt.xlabel(
    "Predicted confidence"
)


plt.ylabel(
    "Actual accuracy"
)


plt.title(
    "OOS Probability Calibration"
)


plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 39. Save
# ============================================================

annual_results.to_csv(

    OUTPUT_DIR
    /
    "annual_calibration_results.csv",

    index=False,
)


all_raw_oos.to_csv(

    OUTPUT_DIR
    /
    "raw_oos_probabilities.csv",

)


all_calibrated_oos.to_csv(

    OUTPUT_DIR
    /
    "calibrated_oos_probabilities.csv",

)


overall_raw_band.to_csv(

    OUTPUT_DIR
    /
    "raw_calibration_band.csv",

    index=False,
)


overall_cal_band.to_csv(

    OUTPUT_DIR
    /
    "calibrated_calibration_band.csv",

    index=False,
)


if calibration_score_rows:

    pd.concat(

        calibration_score_rows,

        ignore_index=True,

    ).to_csv(

        OUTPUT_DIR
        /
        "validation_calibration_method_search.csv",

        index=False,
    )


if calibration_band_frames:

    pd.concat(

        calibration_band_frames,

        ignore_index=True,

    ).to_csv(

        OUTPUT_DIR
        /
        "annual_calibration_bands.csv",

        index=False,
    )


if not confidence_strategy.empty:

    confidence_strategy.to_csv(

        OUTPUT_DIR
        /
        "confidence_profit_table.csv",

        index=False,
    )


if not all_selected_trades.empty:

    all_selected_trades.to_csv(

        OUTPUT_DIR
        /
        "calibrated_strategy_oos_trades.csv",

    )


print()
print("=" * 70)
print("FINISHED")
print("=" * 70)


print(
    OUTPUT_DIR.resolve()
)


print()

print(
    "結果で特に見せてほしい場所:"
)

print(
    "1. ANNUAL CALIBRATION RESULTS"
)

print(
    "2. CALIBRATION METHOD FREQUENCY"
)

print(
    "3. CALIBRATION VALUE"
)

print(
    "4. OVERALL OOS PROBABILITY QUALITY"
)

print(
    "5. RAW CONFIDENCE CALIBRATION"
)

print(
    "6. CALIBRATED CONFIDENCE"
)

print(
    "7. CALIBRATED CONFIDENCE x PROFIT"
)

print(
    "8. CALIBRATION STRATEGY EFFECT"
)

print(
    "9. AUTOMATIC DECISION"
)


## 元セルindex 43


In [ ]:
# ============================================================
# USD/JPY
# ROBUST NESTED PROBABILITY CALIBRATION TEST
#
# 検証:
#   RAW
#   Platt Scaling
#   Isotonic Regression
#
# Entry / Session / 30分固定Exit はこれまでの仕様を維持
#
# ============================================================

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss,
)


# ============================================================
# 0. 必要な変数・関数を確認
# ============================================================

REQUIRED = [
    "data",
    "FEATURES",
    "build_model",
    "predict_frame",
    "choose_threshold",
    "choose_session",
    "select_trades",
    "strategy_stats",
    "BASE_COST",
    "MIN_TRAIN_YEARS",
    "MIN_TRAIN_ROWS",
    "MIN_EVAL_ROWS",
]

missing = [
    x for x in REQUIRED
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "前の戦略コードを先に実行してください。\n"
        f"不足: {missing}"
    )


# ============================================================
# 1. 設定
# ============================================================

EPS = 1e-6

MIN_CAL_ROWS = 5000
MIN_ISOTONIC_ROWS = 10000

CONF_BINS = [
    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
    0.80,
    0.90,
    1.000001,
]

CONF_LABELS = [
    "50-52",
    "52-54",
    "54-56",
    "56-58",
    "58-60",
    "60-62",
    "62-65",
    "65-70",
    "70-80",
    "80-90",
    "90-100",
]

OUTPUT_DIR = (
    Path.cwd()
    / (
        "robust_calibration_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
    )
)

OUTPUT_DIR.mkdir(exist_ok=False)


# ============================================================
# 2. Probability utility
# ============================================================

def clip_prob(p):

    return np.clip(
        np.asarray(
            p,
            dtype=float
        ),
        EPS,
        1 - EPS
    )


def logit(p):

    p = clip_prob(p)

    return np.log(
        p / (1 - p)
    )


# ============================================================
# 3. Calibrator
# ============================================================

class RawCalibrator:

    def fit(self, p, y):
        return self

    def predict(self, p):
        return clip_prob(p)


class PlattCalibrator:

    def __init__(self):

        self.model = LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
        )

    def fit(self, p, y):

        x = logit(
            p
        ).reshape(-1, 1)

        y = np.asarray(
            y,
            dtype=int
        )

        self.model.fit(
            x,
            y
        )

        return self

    def predict(self, p):

        x = logit(
            p
        ).reshape(-1, 1)

        return self.model.predict_proba(
            x
        )[:, 1]


class IsoCalibrator:

    def __init__(self):

        self.model = IsotonicRegression(
            out_of_bounds="clip"
        )

    def fit(self, p, y):

        self.model.fit(
            np.asarray(
                p,
                dtype=float
            ),
            np.asarray(
                y,
                dtype=int
            )
        )

        return self

    def predict(self, p):

        return clip_prob(
            self.model.predict(
                np.asarray(
                    p,
                    dtype=float
                )
            )
        )


# ============================================================
# 4. Calibration後のprobabilityをPredictionへ戻す
# ============================================================

def apply_probability(
    prediction_df,
    p_up,
):

    result = prediction_df.copy()

    p_up = clip_prob(
        p_up
    )

    result["p_up"] = p_up
    result["p_down"] = 1 - p_up

    result["direction"] = np.where(
        result["p_up"]
        >=
        result["p_down"],
        "BUY",
        "SELL"
    )

    result["confidence"] = np.maximum(
        result["p_up"],
        result["p_down"]
    )

    return result


# ============================================================
# 5. ECE
# ============================================================

def calc_ece(
    probability,
    target,
    n_bins=10
):

    probability = clip_prob(
        probability
    )

    target = np.asarray(
        target,
        dtype=int
    )

    edges = np.linspace(
        0,
        1,
        n_bins + 1
    )

    total = len(
        probability
    )

    if total == 0:
        return np.nan

    ece = 0.0

    for i in range(
        n_bins
    ):

        left = edges[i]
        right = edges[i + 1]

        if i == n_bins - 1:

            mask = (
                (probability >= left)
                &
                (probability <= right)
            )

        else:

            mask = (
                (probability >= left)
                &
                (probability < right)
            )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        predicted = probability[
            mask
        ].mean()

        actual = target[
            mask
        ].mean()

        ece += (
            count
            /
            total
            *
            abs(
                predicted
                -
                actual
            )
        )

    return float(
        ece
    )


# ============================================================
# 6. Probability metrics
# ============================================================

def probability_metrics(
    probability,
    target
):

    probability = clip_prob(
        probability
    )

    target = np.asarray(
        target,
        dtype=int
    )

    out = {
        "brier":
            brier_score_loss(
                target,
                probability
            ),

        "ece":
            calc_ece(
                probability,
                target
            ),
    }

    try:

        out["auc"] = roc_auc_score(
            target,
            probability
        )

    except Exception:

        out["auc"] = np.nan

    return out


# ============================================================
# 7. Confidence Band
#
# ★ 前回エラー部分を修正
#
# actual_accuracy列は作らない。
# correct列を直接平均する。
# ============================================================

def calibration_band_table(
    probability,
    target,
):

    probability = clip_prob(
        probability
    )

    target = np.asarray(
        target,
        dtype=int
    )

    predicted_direction = (
        probability >= 0.5
    ).astype(int)

    confidence = np.maximum(
        probability,
        1 - probability
    )

    correct = (
        predicted_direction
        ==
        target
    ).astype(float)

    temp = pd.DataFrame({
        "confidence":
            confidence,

        "correct":
            correct,
    })

    temp["confidence_band"] = pd.cut(
        temp["confidence"],
        bins=CONF_BINS,
        labels=CONF_LABELS,
        right=False,
        include_lowest=True,
    )

    rows = []

    for band, group in temp.groupby(
        "confidence_band",
        observed=True
    ):

        if len(group) == 0:
            continue

        mean_conf = float(
            group[
                "confidence"
            ].mean()
        )

        # ★ここが修正点
        actual_accuracy = float(
            group[
                "correct"
            ].mean()
        )

        rows.append({
            "confidence_band":
                str(
                    band
                ),

            "samples":
                len(
                    group
                ),

            "mean_confidence":
                mean_conf,

            "actual_accuracy":
                actual_accuracy,

            "calibration_gap":
                actual_accuracy
                -
                mean_conf,
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# 8. OOF prediction作成
# ============================================================

def build_oof(
    source_data
):

    source_data = (
        source_data
        .sort_index()
        .copy()
    )

    years = sorted(
        source_data.index.year.unique()
    )

    frames = []

    for year in years:

        year_start = pd.Timestamp(
            year=year,
            month=1,
            day=1,
            tz="UTC"
        )

        year_end = pd.Timestamp(
            year=year + 1,
            month=1,
            day=1,
            tz="UTC"
        )

        train = source_data.loc[
            (
                source_data.index
                <
                year_start
            )
            &
            (
                source_data["label_end"]
                <=
                year_start
            )
        ].copy()

        test = source_data.loc[
            (
                source_data.index
                >=
                year_start
            )
            &
            (
                source_data.index
                <
                year_end
            )
            &
            (
                source_data["label_end"]
                <=
                year_end
            )
        ].copy()

        if (
            len(train)
            <
            MIN_TRAIN_ROWS
        ):
            continue

        if (
            len(test)
            <
            MIN_EVAL_ROWS
        ):
            continue

        model = build_model()

        model.fit(
            train[FEATURES],
            train["target"]
        )

        pred = predict_frame(
            model,
            test
        )

        if "p_up" not in pred.columns:

            raise RuntimeError(
                "predict_frame() に p_up がありません。"
            )

        frame = pd.DataFrame(
            {
                "target":
                    test["target"].astype(int),

                "raw_p_up":
                    pred["p_up"].astype(float),

                "year":
                    year,
            },
            index=test.index
        )

        frames.append(
            frame
        )

    if not frames:

        return pd.DataFrame()

    return (
        pd.concat(
            frames
        )
        .sort_index()
    )


# ============================================================
# 9. Calibrator作成
# ============================================================

def fit_calibrators(
    train_data
):

    oof = build_oof(
        train_data
    )

    calibrators = {
        "RAW":
            RawCalibrator()
    }

    if len(oof) == 0:

        return (
            calibrators,
            oof
        )

    calibrators[
        "RAW"
    ].fit(
        oof["raw_p_up"],
        oof["target"]
    )

    if len(oof) >= MIN_CAL_ROWS:

        platt = PlattCalibrator()

        platt.fit(
            oof["raw_p_up"],
            oof["target"]
        )

        calibrators[
            "PLATT"
        ] = platt

    if len(oof) >= MIN_ISOTONIC_ROWS:

        iso = IsoCalibrator()

        iso.fit(
            oof["raw_p_up"],
            oof["target"]
        )

        calibrators[
            "ISOTONIC"
        ] = iso

    return (
        calibrators,
        oof
    )


# ============================================================
# 10. ValidationでCalibration方式選択
# ============================================================

def choose_calibrator(
    calibrators,
    raw_probability,
    target
):

    rows = []

    for name, calibrator in calibrators.items():

        p = calibrator.predict(
            raw_probability
        )

        metrics = probability_metrics(
            p,
            target
        )

        rows.append({
            "method":
                name,

            **metrics
        })

    table = pd.DataFrame(
        rows
    )

    if table.empty:

        return (
            "RAW",
            table
        )

    table = table.sort_values(
        [
            "brier",
            "ece"
        ],
        ascending=[
            True,
            True
        ]
    )

    selected = str(
        table.iloc[0][
            "method"
        ]
    )

    return (
        selected,
        table
    )


# ============================================================
# 11. Nested Walk Forward
# ============================================================

years = sorted(
    data.index.year.unique()
)

annual_rows = []
method_tables = []

raw_oos_frames = []
cal_oos_frames = []

calibrated_trade_frames = []


for test_year in years:

    validation_year = (
        test_year - 1
    )

    previous_years = [
        y
        for y in years
        if y < validation_year
    ]

    if (
        len(previous_years)
        <
        MIN_TRAIN_YEARS
    ):
        continue

    if (
        validation_year
        not in years
    ):
        continue

    validation_start = pd.Timestamp(
        year=validation_year,
        month=1,
        day=1,
        tz="UTC"
    )

    test_start = pd.Timestamp(
        year=test_year,
        month=1,
        day=1,
        tz="UTC"
    )

    test_end = pd.Timestamp(
        year=test_year + 1,
        month=1,
        day=1,
        tz="UTC"
    )

    train = data.loc[
        (
            data.index
            <
            validation_start
        )
        &
        (
            data["label_end"]
            <=
            validation_start
        )
    ].copy()

    validation = data.loc[
        (
            data.index
            >=
            validation_start
        )
        &
        (
            data.index
            <
            test_start
        )
        &
        (
            data["label_end"]
            <=
            test_start
        )
    ].copy()

    final_train = data.loc[
        (
            data.index
            <
            test_start
        )
        &
        (
            data["label_end"]
            <=
            test_start
        )
    ].copy()

    test = data.loc[
        (
            data.index
            >=
            test_start
        )
        &
        (
            data.index
            <
            test_end
        )
        &
        (
            data["label_end"]
            <=
            test_end
        )
    ].copy()

    if (
        len(train) < MIN_TRAIN_ROWS
        or
        len(validation) < MIN_EVAL_ROWS
        or
        len(final_train) < MIN_TRAIN_ROWS
        or
        len(test) < MIN_EVAL_ROWS
    ):
        continue

    print()
    print("=" * 65)
    print(
        "CALIBRATION TEST YEAR",
        test_year
    )
    print("=" * 65)

    # ========================================================
    # Validation Model
    # ========================================================

    model = build_model()

    model.fit(
        train[FEATURES],
        train["target"]
    )

    val_raw = predict_frame(
        model,
        validation
    )

    # ========================================================
    # Calibration fitting
    # ========================================================

    calibrators, train_oof = fit_calibrators(
        train
    )

    print(
        "Train OOF rows:",
        len(train_oof)
    )

    selected_method, method_table = choose_calibrator(
        calibrators,
        val_raw["p_up"],
        validation["target"]
    )

    method_table[
        "test_year"
    ] = test_year

    method_tables.append(
        method_table
    )

    print()
    print(
        "Calibration comparison"
    )

    print(
        method_table.to_string(
            index=False
        )
    )

    print()
    print(
        "Selected Calibration:",
        selected_method
    )

    # ========================================================
    # ValidationにCalibration適用
    # ========================================================

    val_p = calibrators[
        selected_method
    ].predict(
        val_raw["p_up"]
    )

    val_pred = apply_probability(
        val_raw,
        val_p
    )

    # ========================================================
    # Threshold
    # ========================================================

    threshold_choice, _ = choose_threshold(
        val_pred
    )

    if threshold_choice is None:

        print(
            "Threshold選択失敗"
        )
        continue

    threshold = float(
        threshold_choice[
            "threshold"
        ]
    )

    # ========================================================
    # Session
    # ========================================================

    session_choice, _ = choose_session(
        val_pred,
        threshold
    )

    if session_choice is None:

        print(
            "Session選択失敗"
        )
        continue

    session = session_choice[
        "session_policy"
    ]

    print(
        "Threshold:",
        threshold
    )

    print(
        "Session:",
        session
    )

    # ========================================================
    # Test用 calibrator
    # ========================================================

    test_calibrators, final_oof = fit_calibrators(
        final_train
    )

    if selected_method not in test_calibrators:

        actual_method = "RAW"

    else:

        actual_method = selected_method

    # ========================================================
    # Final Model
    # ========================================================

    final_model = build_model()

    final_model.fit(
        final_train[FEATURES],
        final_train["target"]
    )

    test_raw = predict_frame(
        final_model,
        test
    )

    raw_probability = clip_prob(
        test_raw["p_up"]
    )

    calibrated_probability = (
        test_calibrators[
            actual_method
        ]
        .predict(
            raw_probability
        )
    )

    # ========================================================
    # Probability metrics
    # ========================================================

    raw_metrics = probability_metrics(
        raw_probability,
        test["target"]
    )

    cal_metrics = probability_metrics(
        calibrated_probability,
        test["target"]
    )

    # ========================================================
    # Calibration Band
    #
    # ★ここで前回KeyErrorになっていた
    # ========================================================

    raw_band = calibration_band_table(
        raw_probability,
        test["target"]
    )

    cal_band = calibration_band_table(
        calibrated_probability,
        test["target"]
    )

    # ========================================================
    # Trading
    # ========================================================

    calibrated_test = apply_probability(
        test_raw,
        calibrated_probability
    )

    raw_trades = select_trades(
        test_raw,
        threshold=threshold,
        session_policy=session,
        cost=BASE_COST
    )

    cal_trades = select_trades(
        calibrated_test,
        threshold=threshold,
        session_policy=session,
        cost=BASE_COST
    )

    if len(raw_trades) > 0:

        raw_stats = strategy_stats(
            raw_trades[
                "net_return"
            ]
        )

    else:

        raw_stats = {
            "profit_factor": np.nan,
            "avg_return": np.nan,
            "max_dd": np.nan,
        }

    if len(cal_trades) > 0:

        cal_stats = strategy_stats(
            cal_trades[
                "net_return"
            ]
        )

    else:

        cal_stats = {
            "profit_factor": np.nan,
            "avg_return": np.nan,
            "max_dd": np.nan,
        }

    # ========================================================
    # OOS 保存
    # ========================================================

    raw_oos = pd.DataFrame(
        {
            "target":
                test["target"],

            "p_up":
                raw_probability,

            "test_year":
                test_year,
        },
        index=test.index
    )

    cal_oos = pd.DataFrame(
        {
            "target":
                test["target"],

            "p_up":
                calibrated_probability,

            "method":
                actual_method,

            "test_year":
                test_year,
        },
        index=test.index
    )

    raw_oos_frames.append(
        raw_oos
    )

    cal_oos_frames.append(
        cal_oos
    )

    if len(cal_trades) > 0:

        temp = cal_trades.copy()

        temp["test_year"] = (
            test_year
        )

        temp["calibration_method"] = (
            actual_method
        )

        calibrated_trade_frames.append(
            temp
        )

    annual_rows.append({
        "test_year":
            test_year,

        "method":
            actual_method,

        "threshold":
            threshold,

        "session":
            session,

        "raw_brier":
            raw_metrics[
                "brier"
            ],

        "cal_brier":
            cal_metrics[
                "brier"
            ],

        "raw_ece":
            raw_metrics[
                "ece"
            ],

        "cal_ece":
            cal_metrics[
                "ece"
            ],

        "raw_auc":
            raw_metrics[
                "auc"
            ],

        "cal_auc":
            cal_metrics[
                "auc"
            ],

        "raw_trades":
            len(
                raw_trades
            ),

        "cal_trades":
            len(
                cal_trades
            ),

        "raw_pf":
            raw_stats[
                "profit_factor"
            ],

        "cal_pf":
            cal_stats[
                "profit_factor"
            ],

        "raw_avg":
            raw_stats[
                "avg_return"
            ],

        "cal_avg":
            cal_stats[
                "avg_return"
            ],

        "raw_dd":
            raw_stats[
                "max_dd"
            ],

        "cal_dd":
            cal_stats[
                "max_dd"
            ],
    })

    print()
    print(
        "RAW Brier:",
        raw_metrics["brier"]
    )

    print(
        "CAL Brier:",
        cal_metrics["brier"]
    )

    print(
        "RAW ECE:",
        raw_metrics["ece"]
    )

    print(
        "CAL ECE:",
        cal_metrics["ece"]
    )

    print(
        "RAW PF:",
        raw_stats[
            "profit_factor"
        ]
    )

    print(
        "CAL PF:",
        cal_stats[
            "profit_factor"
        ]
    )


# ============================================================
# 12. Annual Results
# ============================================================

annual_results = pd.DataFrame(
    annual_rows
)

if annual_results.empty:

    raise RuntimeError(
        "評価年が1年も作成されませんでした。"
    )

print()
print("=" * 70)
print("ANNUAL CALIBRATION RESULTS")
print("=" * 70)

print(
    annual_results.to_string(
        index=False
    )
)


# ============================================================
# 13. Method frequency
# ============================================================

print()
print("=" * 70)
print("CALIBRATION METHOD FREQUENCY")
print("=" * 70)

print(
    annual_results[
        "method"
    ]
    .value_counts()
)


# ============================================================
# 14. Improvement
# ============================================================

annual_results[
    "brier_improved"
] = (
    annual_results[
        "cal_brier"
    ]
    <
    annual_results[
        "raw_brier"
    ]
)

annual_results[
    "ece_improved"
] = (
    annual_results[
        "cal_ece"
    ]
    <
    annual_results[
        "raw_ece"
    ]
)

annual_results[
    "pf_improved"
] = (
    annual_results[
        "cal_pf"
    ]
    >
    annual_results[
        "raw_pf"
    ]
)

annual_results[
    "avg_improved"
] = (
    annual_results[
        "cal_avg"
    ]
    >
    annual_results[
        "raw_avg"
    ]
)


print()
print("=" * 70)
print("CALIBRATION VALUE")
print("=" * 70)

n_years = len(
    annual_results
)

print(
    "Brier improved:",
    annual_results[
        "brier_improved"
    ].sum(),
    "/",
    n_years
)

print(
    "ECE improved:",
    annual_results[
        "ece_improved"
    ].sum(),
    "/",
    n_years
)

print(
    "PF improved:",
    annual_results[
        "pf_improved"
    ].sum(),
    "/",
    n_years
)

print(
    "Avg improved:",
    annual_results[
        "avg_improved"
    ].sum(),
    "/",
    n_years
)


# ============================================================
# 15. Overall OOS
# ============================================================

all_raw = pd.concat(
    raw_oos_frames
).sort_index()

all_cal = pd.concat(
    cal_oos_frames
).sort_index()


raw_overall = probability_metrics(
    all_raw["p_up"],
    all_raw["target"]
)

cal_overall = probability_metrics(
    all_cal["p_up"],
    all_cal["target"]
)


print()
print("=" * 70)
print("OVERALL OOS PROBABILITY QUALITY")
print("=" * 70)

print(
    "RAW:",
    raw_overall
)

print(
    "CAL:",
    cal_overall
)


# ============================================================
# 16. Confidence calibration
# ============================================================

raw_conf_table = calibration_band_table(
    all_raw["p_up"],
    all_raw["target"]
)

cal_conf_table = calibration_band_table(
    all_cal["p_up"],
    all_cal["target"]
)


print()
print("=" * 70)
print("RAW CONFIDENCE CALIBRATION")
print("=" * 70)

raw_show = raw_conf_table.copy()

for col in [
    "mean_confidence",
    "actual_accuracy",
    "calibration_gap",
]:

    if col in raw_show.columns:
        raw_show[col] *= 100

print(
    raw_show.to_string(
        index=False
    )
)


print()
print("=" * 70)
print("CALIBRATED CONFIDENCE")
print("=" * 70)

cal_show = cal_conf_table.copy()

for col in [
    "mean_confidence",
    "actual_accuracy",
    "calibration_gap",
]:

    if col in cal_show.columns:
        cal_show[col] *= 100

print(
    cal_show.to_string(
        index=False
    )
)


# ============================================================
# 17. Confidence × Profit
# ============================================================

if calibrated_trade_frames:

    all_trades = pd.concat(
        calibrated_trade_frames
    ).sort_index()

    all_trades[
        "confidence_band"
    ] = pd.cut(
        all_trades[
            "confidence"
        ],
        bins=CONF_BINS,
        labels=CONF_LABELS,
        right=False,
        include_lowest=True,
    )

    profit_rows = []

    for band, group in all_trades.groupby(
        "confidence_band",
        observed=True
    ):

        if len(group) == 0:
            continue

        stats = strategy_stats(
            group[
                "net_return"
            ]
        )

        profit_rows.append({
            "confidence_band":
                str(
                    band
                ),

            "trades":
                len(
                    group
                ),

            "mean_confidence":
                group[
                    "confidence"
                ].mean(),

            **stats
        })

    confidence_profit = pd.DataFrame(
        profit_rows
    )

    print()
    print("=" * 70)
    print("CALIBRATED CONFIDENCE x PROFIT")
    print("=" * 70)

    show = confidence_profit.copy()

    for col in [
        "mean_confidence",
        "win_rate",
        "avg_return",
        "median_return",
        "max_dd",
        "growth",
    ]:

        if col in show.columns:
            show[col] *= 100

    print(
        show.to_string(
            index=False
        )
    )

else:

    confidence_profit = pd.DataFrame()


# ============================================================
# 18. Automatic Decision
# ============================================================

print()
print("=" * 70)
print("AUTOMATIC DECISION")
print("=" * 70)

print(
    "RAW Brier:",
    raw_overall[
        "brier"
    ]
)

print(
    "CAL Brier:",
    cal_overall[
        "brier"
    ]
)

print(
    "RAW ECE:",
    raw_overall[
        "ece"
    ]
)

print(
    "CAL ECE:",
    cal_overall[
        "ece"
    ]
)

print()

if (
    cal_overall["brier"]
    <
    raw_overall["brier"]

    and

    cal_overall["ece"]
    <
    raw_overall["ece"]

    and

    annual_results[
        "brier_improved"
    ].sum()
    >=
    4
):

    print(
        "判定: CALIBRATION HAS CLEAR OOS VALUE"
    )

    print(
        "→ Calibration後ConfidenceをPosition Sizing候補にします。"
    )

else:

    print(
        "判定: RAW PROBABILITY REMAINS PREFERABLE"
    )

    print(
        "→ 無理にCalibrationを採用しません。"
    )


# ============================================================
# 19. Calibration Graph
# ============================================================

plt.figure(
    figsize=(
        7,
        6
    )
)

plt.plot(
    [0.5, 1.0],
    [0.5, 1.0],
    linestyle="--"
)

if not raw_conf_table.empty:

    plt.plot(
        raw_conf_table[
            "mean_confidence"
        ],
        raw_conf_table[
            "actual_accuracy"
        ],
        marker="o",
        label="RAW"
    )

if not cal_conf_table.empty:

    plt.plot(
        cal_conf_table[
            "mean_confidence"
        ],
        cal_conf_table[
            "actual_accuracy"
        ],
        marker="o",
        label="CALIBRATED"
    )

plt.xlabel(
    "Predicted Confidence"
)

plt.ylabel(
    "Actual Accuracy"
)

plt.title(
    "Nested OOS Probability Calibration"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 20. Save
# ============================================================

annual_results.to_csv(
    OUTPUT_DIR
    /
    "annual_calibration.csv",
    index=False
)

raw_conf_table.to_csv(
    OUTPUT_DIR
    /
    "raw_confidence.csv",
    index=False
)

cal_conf_table.to_csv(
    OUTPUT_DIR
    /
    "calibrated_confidence.csv",
    index=False
)

all_raw.to_csv(
    OUTPUT_DIR
    /
    "raw_oos.csv"
)

all_cal.to_csv(
    OUTPUT_DIR
    /
    "calibrated_oos.csv"
)

if method_tables:

    pd.concat(
        method_tables,
        ignore_index=True
    ).to_csv(
        OUTPUT_DIR
        /
        "validation_method_selection.csv",
        index=False
    )

if not confidence_profit.empty:

    confidence_profit.to_csv(
        OUTPUT_DIR
        /
        "confidence_profit.csv",
        index=False
    )


print()
print("=" * 70)
print("FINISHED")
print("=" * 70)

print(
    OUTPUT_DIR.resolve()
)

print()
print(
    "次にスクショしてほしい場所:"
)

print(
    "1. ANNUAL CALIBRATION RESULTS"
)

print(
    "2. CALIBRATION METHOD FREQUENCY"
)

print(
    "3. CALIBRATION VALUE"
)

print(
    "4. OVERALL OOS PROBABILITY QUALITY"
)

print(
    "5. RAW CONFIDENCE CALIBRATION"
)

print(
    "6. CALIBRATED CONFIDENCE"
)

print(
    "7. CALIBRATED CONFIDENCE x PROFIT"
)

print(
    "8. AUTOMATIC DECISION"
)
